# Chapter 1 — PromptTemplate exploration

My own experiments applying the ideas from Chapter 1 of *AI Agents and Applications*.

Goal: build intuition for `PromptTemplate` and the `prompt | llm` chain pattern using **my own** examples rather than the book's Segovia text.

---

## Setup

In [3]:
from dotenv import load_dotenv
load_dotenv()  # loads OPENAI_API_KEY from .env

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

## Experiment 1 — Plain f-string vs PromptTemplate

Sanity check that they produce the same output. Then we'll see why PromptTemplate is the better starting point even when they look equivalent.

In [5]:
text = """Pickleball is a paddle sport invented in 1965 on Bainbridge Island, Washington.
It blends elements of tennis, badminton, and table tennis, and is played on a court roughly
the size of a badminton court with a perforated polymer ball and solid paddles."""

# Version A — f-string
n_words, tone = 15, "witty"
fstring_prompt = f"You are a copywriter. Write a {n_words}-word summary in a {tone} tone: {text}"
print("--- f-string version ---")
print(llm.invoke(fstring_prompt).content)

--- f-string version ---
Pickleball: where tennis, badminton, and table tennis had a wild baby on Bainbridge Island!


In [6]:
# Version B — PromptTemplate + chain
summarize = PromptTemplate.from_template(
    "You are a copywriter. Write a {n_words}-word summary in a {tone} tone: {text}"
)
chain = summarize | llm

print("--- chain version ---")
print(chain.invoke({"text": text, "n_words": 15, "tone": "witty"}).content)

--- chain version ---
Pickleball: where tennis meets badminton for a paddle party on a petite court—pickled perfection!


## Experiment 2 — Reuse the same template, vary the tone

The whole point of templates: define once, call many times. Loop over tones.

In [7]:
for tone in ["clinical", "playful", "ominous", "academic"]:
    out = chain.invoke({"text": text, "n_words": 20, "tone": tone})
    print(f"[{tone}] {out.content}\n")

[clinical] Pickleball, invented in 1965 on Bainbridge Island, integrates tennis, badminton, and table tennis elements, utilizing paddles and a polymer ball.

[playful] Pickleball: where tennis, badminton, and table tennis have a paddle party! Invented in '65, it's the coolest court craze!

[ominous] In 1965, on Bainbridge Island, a new sport emerged—a chilling fusion of competition, echoing with the specter of pickleball's rise.

[academic] Invented in 1965 on Bainbridge Island, Washington, pickleball integrates aspects of tennis, badminton, and table tennis, utilizing specific equipment.



## Experiment 3 — Two-step chain

Chain a translation step into a summarization step. The output of step 1 becomes the input to step 2.

Not the cleanest LCEL composition yet — that comes in later chapters — but it shows the idea.

In [8]:
translate = PromptTemplate.from_template("Translate to {language}: {text}")
translate_chain = translate | llm

french = translate_chain.invoke({"language": "French", "text": text}).content
print("French translation:", french, "\n")

fr_summary = chain.invoke({"text": french, "n_words": 20, "tone": "playful"}).content
print("Summary of French version (still asks for English output by default):", fr_summary)

French translation: Le pickleball est un sport de raquette inventé en 1965 sur l'île Bainbridge, dans l'État de Washington. Il combine des éléments de tennis, de badminton et de tennis de table, et se joue sur un court d'environ la taille d'un court de badminton avec une balle en polyéthylène perforée et des raquettes solides. 

Summary of French version (still asks for English output by default): Le pickleball, né en 1965 à Bainbridge, mixe tennis, badminton et ping-pong sur un petit court. Prêt à frapper ? 🎾✨


## Takeaways

- The output of `chain.invoke({...})` is an `AIMessage` object — use `.content` to get the string.
- The `|` operator works because each piece (`PromptTemplate`, `ChatOpenAI`) implements LangChain's `Runnable` interface. That's the whole secret.
- Switching from f-strings to `PromptTemplate` adds ~2 lines of code and unlocks: looping over variations, A/B testing, prompt versioning, loading prompts from files.

Up next: Chapter 2 — prompt engineering techniques.